In [ ]:
# Figure 7
# Distribution of twins for projections sampled by constructing uniform random
# hypergraphs for different settings of (k, n, m)

# Reads compressed results and automatically reports truly missing data.

In [ ]:
import math
import pathlib
import traceback

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import (data_exists, read_data,
                            read_sampled_cell, report_excluded_cells)

In [ ]:
title_size = 18
label_size = 17
tick_size = 15
legend_font = 15
labelpad = 3
mpl.rcParams["axes.titlesize"] = title_size
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size

In [ ]:
m_vals = list(range(2, 21))

In [ ]:
results_base = pathlib.Path("../../results/reproduce/")
heatmap_results_path = results_base / "heatmap-1000/"

In [ ]:
heatmap_infos = []
num_samples = 1000
k_vals = [3, 4]
min_n = 6
max_n = 16
n_vals = list(range(min_n, max_n+1))

min_m = 6
max_m = 16
m_vals = list(range(min_m, max_m+1))

mats = {k: np.zeros((len(m_vals), len(n_vals))) for k in k_vals}
for k in k_vals:
    for m in m_vals:
        for n in n_vals:
            filename = heatmap_results_path / f"n-{n}_m-{m}_k-{k}_samples-{num_samples}.csv"
            output_data, info = read_sampled_cell(
                filename, num_samples)
            heatmap_infos.append(info)
            if output_data is not None:
                num_iso_classes = np.array(output_data["total_filtered"])
                num_multi_iso = np.sum(num_iso_classes > 1)
                # Divide by the rows actually read, not the nominal sample
                # count: identical for a complete cell, and this cannot
                # silently deflate a reduced-N one.
                mats[k][m-min_m, n-min_n] = num_multi_iso / info["rows"]
            else:
                # Absent is expected where m*k < n (cannot span n nodes) or
                # m = C(n,k) (a single hypergraph, no twins).
                mats[k][m-min_m, n-min_n] = 0.0

report_excluded_cells(heatmap_infos, "figure 7 heatmap")

In [ ]:
sampled_results_path = results_base / "density-500/"

In [ ]:
fig = plt.figure(figsize=(12, 7), constrained_layout=True)

gs = fig.add_gridspec(2, 2, width_ratios=[0.5, 0.5], wspace=0.1)
num_samples=1000
min_n = 6
max_n = 16
n_vals = list(range(min_n, max_n+1))

min_m = 6
max_m = 16
m_vals = list(range(min_m, max_m+1))
# Left: Heatmaps showing proportion of sampled projection
# matrices with at least 2 isomorphism classes in their set of twins
vmin = 0
vmax = 0
for k in k_vals:
    if mats[k].min() < vmin:
        vmin = mats[k].min()
    if mats[k].max() > vmax:
        vmax = mats[k].max()

for row, k in enumerate(k_vals):
    ax = fig.add_subplot(gs[row, 0])
    im = ax.imshow(mats[k], origin="lower", cmap="Oranges", vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax)
    ax.set_title(rf"$k={k}$")
    ax.set_xlabel(r"$n$")
    ax.set_yticks(list(range(len(m_vals)))[::2], m_vals[::2])
    ax.set_xticks(list(range(len(m_vals)))[::2], m_vals[::2])
    ax.set_ylabel(r"$m$")


n_colors = {
    6: "#0571b0",
    7: "#a6dba0",
    8: "#7b3294",
    9: "#fdae61"
}

num_samples = 500
max_ns = {3: 9, 4: 8}
line_infos = []
for row, k in enumerate([3, 4]):
    ax = fig.add_subplot(gs[row, 1])
    for n in range(6, max_ns[k]+1):
        m_vals = list(range(2, math.comb(n, k)))
        mean_isos_m = np.zeros(len(m_vals))
        n_samples_m = np.zeros(len(m_vals))
        std_isos_m = np.zeros(len(m_vals))
        for midx, m in enumerate(m_vals):
            filename = sampled_results_path / f"n-{n}_m-{m}_k-{k}_samples-{num_samples}.csv"
            try:
                output_data, info = read_sampled_cell(
                    filename, num_samples)
            except Exception:
                traceback.print_exc()
                print(filename)
                continue
            line_infos.append(info)
            if output_data is not None:
                num_iso_classes = np.array(output_data["total_filtered"])
                mean_isos_m[midx] = num_iso_classes.mean()
                std_isos_m[midx] = num_iso_classes.std()
                n_samples_m[midx] = info["rows"]
        
        mean_isos_m[mean_isos_m == 0] = np.nan
        std_isos_m[mean_isos_m == 0] = np.nan
        ax.plot(m_vals, mean_isos_m, label=rf"${n}$", color=n_colors[n])
        ax.fill_between(m_vals, mean_isos_m, mean_isos_m+std_isos_m, alpha=0.4, color=n_colors[n])
        
    ax.legend(frameon=False)
    ax.set_yscale("log")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(r"$m$")
    ax.set_ylabel("Avg. Iso. Classes")
    xticks = list(range(math.comb(n, k)+1))[0::5]
    ax.set_xticks(xticks, xticks)

fig.savefig("../plots/uniform-combined-updated.pdf", bbox_inches="tight")

report_excluded_cells(line_infos, "figure 7 line panel")